# Proyecto

Minería de Datos \
CC5205-2 - Primavera 2025 \
Integrantes: 
- Sebastián Cadena
- Juan Balboa
- Benjamín Fuentes Rodrigo
- Camila Rojas
- Matías Méndez


## Dataset.

In [267]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit, StratifiedKFold, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from sklearn.feature_selection import SelectKBest, SelectFromModel, mutual_info_classif

# import matplotlib.pyplot as plt

Dataset utilizado: **114000 Spotify Songs**. \
Obtenido de: https://www.kaggle.com/datasets/priyamchoksi/spotify-dataset-114k-songs.

In [268]:
# Se leen y muestran los datos.
spotifySongDS = pd.read_csv('dataset.csv')
spotifySongDS.head(5)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [269]:
# Se muestran las columnas del dataset a partir de un ejemplo.
ej = spotifySongDS.iloc[10]
print(ej)

Unnamed: 0                              10
track_id            4mzP5mHkRvGxdhdGdAH7EJ
artists                       Zack Tabudlo
album_name                         Episode
track_name            Give Me Your Forever
popularity                              74
duration_ms                         244800
explicit                             False
danceability                         0.627
energy                               0.363
key                                      8
loudness                            -8.127
mode                                     1
speechiness                         0.0291
acousticness                         0.279
instrumentalness                       0.0
liveness                            0.0928
valence                              0.301
tempo                               99.905
time_signature                           4
track_genre                       acoustic
Name: 10, dtype: object


## Clasificador de géneros en canciones.

In [ ]:
# Se crea una copia del dataset.
classificationData = spotifySongDS.copy()

# Se elige el número de clases.
classes = list(range(0, 100, 10)) # list(range(0, 100))
topgenre = classificationData.groupby("track_genre").count().index[classes]
print(f"Clases utilizadas: {topgenre}")

# Se filtran los datos según las clases elegidas.
classificationData = classificationData[classificationData["track_genre"].isin(topgenre)]

Clases utilizadas: Index(['acoustic', 'breakbeat', 'dance', 'edm', 'gospel', 'heavy-metal',
       'j-dance', 'mandopop', 'pop', 'rock'],
      dtype='object', name='track_genre')


In [271]:
# Se muestran las columnas disponibles.
classificationData.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

In [272]:
# Características utilizadas.
c = ['popularity', 'duration_ms', 'explicit', 'danceability', "energy",
     "key", "loudness", "mode", "speechiness", "acousticness", 
     'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

X_clasification = classificationData[c]

# Clase.
y_clasification_str = np.squeeze(classificationData[["track_genre"]])
le = LabelEncoder()
le.fit(y_clasification_str)
y_clasification = le.transform(y_clasification_str)
# le.inverse_transform()

In [273]:
# Se escala los datos.
scalerC = StandardScaler()
X_clasification_scaled = scalerC.fit_transform(X_clasification)

In [274]:
# Se separan los conjuntos X e y en conjuntos de entrenamiento, validación y prueba.
X_train, X_vt, y_train, y_vt = train_test_split(X_clasification_scaled, y_clasification, test_size = 0.4, random_state = 0, stratify = y_clasification)
X_val, X_test, y_val, y_test = train_test_split(X_vt, y_vt, test_size = 0.5, random_state = 0)

In [275]:
# Se elijen los mejores parámetros para clasificar con SelectKbest.
kBestModel = SelectKBest(mutual_info_classif, k = 11)
kBestModel.fit(X_train, y_train)
X_train_k = kBestModel.transform(X_train)
X_val_k = kBestModel.transform(X_val)

kBestModel.get_feature_names_out(input_features = c)

array(['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo'], dtype=object)

In [285]:
# Se concatena el conjunto de entrenamiento con el de validación.
X = np.concatenate((X_train_k, X_val_k))
Y = np.concatenate((y_train, y_val))

# Se crea el modelo de Random Forest.
rf = RandomForestClassifier(random_state = 0)

# Se crea el diccionario de hiperparámetros.
hiperparam_Random_Forest = {"max_depth": [None, 5, 15, 20, 25, 35], "n_estimators": [300, 400, 450, 500, 600]}

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 0)
#cv = StratifiedShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 0)

In [286]:
# Creamos la grilla para evaluar los hiperparámetros
grid_Random_Forest = GridSearchCV(rf, hiperparam_Random_Forest, cv = cv, verbose = 5)

# Entrenamos cada caso con el conjunto de entrenamiento y validación
grid_Random_Forest.fit(X, Y)

# Hiperparámetro del mejor clasificador Random Forest, con su puntación
print("\nLos parámetros del mejor clasificador Random Forest, según la grilla son:")
print(grid_Random_Forest.best_params_)
print("\nCon una puntación de:", grid_Random_Forest.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV 1/5] END ..max_depth=None, n_estimators=300;, score=0.736 total time=   6.0s
[CV 2/5] END ..max_depth=None, n_estimators=300;, score=0.734 total time=   5.9s
[CV 3/5] END ..max_depth=None, n_estimators=300;, score=0.744 total time=   6.0s
[CV 4/5] END ..max_depth=None, n_estimators=300;, score=0.736 total time=   6.5s
[CV 5/5] END ..max_depth=None, n_estimators=300;, score=0.749 total time=   6.6s
[CV 1/5] END ..max_depth=None, n_estimators=400;, score=0.737 total time=   8.9s
[CV 2/5] END ..max_depth=None, n_estimators=400;, score=0.734 total time=   8.2s
[CV 3/5] END ..max_depth=None, n_estimators=400;, score=0.749 total time=   8.2s
[CV 4/5] END ..max_depth=None, n_estimators=400;, score=0.733 total time=   8.4s
[CV 5/5] END ..max_depth=None, n_estimators=400;, score=0.754 total time=   8.2s
[CV 1/5] END ..max_depth=None, n_estimators=450;, score=0.738 total time=   9.6s
[CV 2/5] END ..max_depth=None, n_estimators=450

In [287]:
# Se crea el modelo definitivo :O
rfFinal = RandomForestClassifier(max_depth = 15, n_estimators = 300, random_state = 0) 
rfFinal.fit(X_train_k, y_train)

# Casos buenos:
# 20, 450, con 11 parámetros.
# 15, 300, 11 parámetros.

c_val_pred = rfFinal.predict(X_val_k)

In [288]:
print(classification_report(c_val_pred, y_val, digits=3))

              precision    recall  f1-score   support

           0      0.566     0.716     0.632       155
           1      0.904     0.867     0.885       196
           2      0.505     0.580     0.540       169
           3      0.616     0.584     0.600       214
           4      0.812     0.804     0.808       194
           5      0.891     0.849     0.870       212
           6      0.816     0.816     0.816       206
           7      0.678     0.634     0.656       216
           8      0.701     0.600     0.647       250
           9      0.739     0.798     0.767       188

    accuracy                          0.723      2000
   macro avg      0.723     0.725     0.722      2000
weighted avg      0.729     0.723     0.724      2000

